# C10-competition-craft — Practice p14 — Solution

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804
SIGNAL = ["honey_stores_kg", "autumn_hive_mass_kg", "varroa_mite_index",
          "forager_traffic_per_min", "brood_frames", "daily_temp_swing_c",
          "queen_age_years"]
ks = np.array([7, 9, 11])

df = pd.read_csv("../data/train.csv")
FEATURES = [c for c in df.columns if c != "outcome"]
X = df[FEATURES]
y = df["outcome"].to_numpy()

dead_cells = [3, 6, 8]

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=150, random_state=SEED, stratify=y
)
scores = []
for k in ks:
    candidate = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=int(k))),
    ]).fit(X_tr[SIGNAL], y_tr)
    scores.append(f1_score(y_val, candidate.predict(X_val[SIGNAL]), average="macro"))
val_f1s = np.array(scores, dtype=float)
best_k = int(ks[np.argmax(val_f1s)])

final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k)),
]).fit(X[SIGNAL], y)


def predict_labels(X_test):
    return pd.Series(final_pipe.predict(X_test[SIGNAL]), index=X_test.index)


probe = X.iloc[250:290]
probe_out = predict_labels(probe)
contract_ok = bool(
    isinstance(probe_out, pd.Series)
    and len(probe_out) == len(probe)
    and probe_out.index.equals(probe.index)
    and set(probe_out.unique()) <= set(np.unique(y))
)


def run_submission():
    local_features = [c for c in df.columns if c != "outcome"]
    local_X = df[local_features]
    local_y = df["outcome"].to_numpy()
    local_tr, local_val, local_y_tr, local_y_val = train_test_split(
        local_X, local_y, test_size=150, random_state=SEED, stratify=local_y
    )
    local_scores = []
    for local_k in ks:
        local_candidate = Pipeline([
            ("scaler", StandardScaler()),
            ("knn", KNeighborsClassifier(n_neighbors=int(local_k))),
        ]).fit(local_tr[SIGNAL], local_y_tr)
        local_scores.append(
            f1_score(local_y_val, local_candidate.predict(local_val[SIGNAL]), average="macro")
        )
    local_val_f1s = np.array(local_scores, dtype=float)
    local_best_k = int(ks[np.argmax(local_val_f1s)])
    local_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=local_best_k)),
    ]).fit(local_X[SIGNAL], local_y)

    def local_predict_labels(X_test):
        return pd.Series(local_pipe.predict(X_test[SIGNAL]), index=X_test.index)

    preds = local_predict_labels(local_X.iloc[250:290]).to_numpy()
    return local_best_k, float(local_val_f1s[-1]), preds


audit_a = run_submission()
audit_b = run_submission()
audit_ok = bool(
    audit_a[0] == audit_b[0]
    and audit_a[1] == audit_b[1]
    and np.array_equal(audit_a[2], audit_b[2])
)
(dead_cells, val_f1s, best_k, contract_ok, audit_ok)

- Cell 3 breaks D3: the abandoned file read raises `FileNotFoundError` and stops the fresh run.
- Cell 6 breaks D3: the stale column name raises `KeyError` before the final model is reached.
- Cell 8 breaks D3 (and exposes a D2 stale-global dependency): `best_model_old` is absent in a fresh kernel, so `NameError` stops the run.

### Answer check

In [ ]:
assert dead_cells == [3, 6, 8]
expected = np.array([0.8119212962962963, 0.8054661607185745,
                     0.8198760747394207])
assert val_f1s.shape == (3,)
assert np.allclose(val_f1s, expected, atol=1e-12, rtol=0)
assert best_k == 11
assert contract_ok is True
assert audit_a[0] == best_k and audit_a[1] == val_f1s[-1]
assert audit_ok is True